Sklearn

In [ ]:
from sklearn.model_selection import train_test_split
import torch

# Split
X_train,X_test,y_train,y_test = \
train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Accuracy (TP+TN)/Total
accuracy_score(y_true,y_pred)

# Precision  TP/(TP+FP)
precision_score(y_true,y_pred)


# Recall   TP/(TP+FN)
recall_score(y_true,y_pred)


# F1- 2PR/(P+R)
f1_score(y_true,y_pred)


# without sklearn
y_true = torch.tensor([
1,0,1,1,0,0,1,0
])

y_prob = torch.tensor([
0.9,0.7,0.8,0.3,
0.1,0.4,0.95,0.2
])

## if this numpy array is a binary mask, you can compute the metrics as follows:
# y = (predictions > threshold).astype(int)
# python list:
# y = [1 if p > threshold else 0 for p in predictions]

y_pred_binary = (y_prob > 0.5).float()
accuracy  = (y_pred_binary == y_true).float().mean()

TP = ((y_pred_binary == y_true) & (y_true == 1)).sum()
FP = ((y_pred_binary != y_true) & (y_true == 0)).sum()

Precision = TP / (TP + FP)

FN = ((y_pred_binary != y_true) & (y_true == 1)).sum()
Recall = TP / (TP + FN)

F1 = 2 * (Precision * Recall) / (Precision + Recall)

# IOU
# Intersection over Union (IoU) and Dice Coefficient for binary masks
IOU = TP / (TP + FP + FN)

# for masks with numpy

import numpy as np
mask1 = np.array([
    [1, 1, 0],
    [0, 1, 0]
])

mask2 = np.array([
    [1, 0, 0],
    [1, 1, 0]
])

intersection = np.logical_and(mask1, mask2).sum()

union = np.logical_or(mask1, mask2).sum()

iou = intersection / union

# Dice Coefficient
Dice = 2 * TP / (2 * TP + FP + FN)  # binary case
dice = 2 * intersection / (mask1.sum() + mask2.sum())



PyTorch Tensors

In [ ]:
import torch
from torchvision import models, datasets, transforms

x = torch.tensor([1,2,3])

x.shape

x.dtype

x.device

# GPU
device = torch.device("cuda"if torch.cuda.is_available()else "cpu")

model.to(device)

x = x.to(device)

# create a dataset
from torch.utils.data import Dataset,DataLoader

class MyDataset(Dataset):

    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        if self.transform:
            x = self.transform(x)

        return x, y
    
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = MyDataset(X_train,y_train, transform=transform)    
# DataLoader - creates batches of data and shuffles them at each epoch.

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

CNN binary model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Input: [B, 3, 224, 224]
# Formula:
# H_out = (H + 2P - K)/S + 1


class BinaryCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1) # Output: [B, 32, 224, 224]
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)  
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1) 
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2) 
        self.adaptive_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(128, 1)
        self.dropout = nn.Dropout(0.5)

        # Binary classification: Output shape = [B,1]

    def forward(self, x):

        # input x = [B,3,224,224]

        x = F.relu(self.bn1(self.conv1(x))) 
        x = self.pool(x) # Output: [B, 64, 112, 112]

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x) # Output: [B, 128, 56, 56]

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x) # Output: [B, 128, 28, 28]

        x = self.adaptive_pool(x) # Output: [B, 128, 1, 1]
        x = torch.flatten(x, 1) # Output: [B, 128]

        x = self.dropout(x)
        logits = self.fc(x)   # output: [B,1]

        return logits
    
# Training Pipeline

model = BinaryCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

num_epochs = 10
for epoch in range(num_epochs):

    model.train()

    for images, labels in train_loader:

        images = images.to(device)

        # BCE expects float labels
        labels = labels.float().unsqueeze(1).to(device)
        # [B] -> [B,1]

        optimizer.zero_grad() # zero the gradients on each iteration

        logits = model(images) # [B,1] # compute output logits

        loss = criterion(logits, labels) # compute loss

        loss.backward() # compute gradients dL/dW

        optimizer.step() # update weights W = W - lr * gradient
        
# Inference
model.eval()

with torch.no_grad():

    logits = model(images)

    probs = torch.sigmoid(logits)
    # convert logits -> probability

    preds = (probs > 0.5).float()

Data leakage images

In [ ]:
import imagehash
from PIL import Image

hashes = {}

for img_path in image_paths:
    h = imagehash.phash(Image.open(img_path))

    if h in hashes:
        print("Duplicate found:", img_path)
    else:
        hashes[h] = img_path